# Setup

## Import Packages and Functions

In [ ]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings("ignore")
import sys

from dataset import TrainDataset, TestDataset
from preprocessing import vjepa_preprocessing
from create_submission import encode_depth, save_submission

## Define Globals and Hyperparameters

In [ ]:
TRAIN_DIR = '../data/train'
TEST_DIR = '../data/test'
DEPTH_ANYTHING_MODEL = 'vitb' # vits, vitb, vitl, vitg
BATCH_SIZE = 4

## Load Train and Test Data

In [ ]:
train_dataset = TrainDataset(TRAIN_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TestDataset(TEST_DIR)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

### Visualize Training Data

In [ ]:
batch = next(iter(train_loader))
images = batch["image"]
depths = batch["depth"]

print("Images:", images.shape)
print("Depths:", depths.shape)

_, axes = plt.subplots(2, BATCH_SIZE, figsize=(4 * BATCH_SIZE, 8))
for i in range(BATCH_SIZE):
    img = images[i].permute(1, 2, 0) / 255.0
    depth = depths[i].squeeze(0)
    axes[0, i].imshow(img)
    axes[0, i].set_title("RGB image")
    axes[0, i].axis("off")
    im = axes[1, i].imshow(depth, cmap='viridis')
    axes[1, i].set_title('Depth map')
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

# V-JEPA Encoding

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)
print("original images:", images.shape)

images_vjepa = vjepa_preprocessing(images)
print("V-JEPA input:", images_vjepa.shape)

encoder, _ = torch.hub.load('../external/vjepa2',
                            'vjepa2_1_vit_large_384',
                            source = 'local')

encoder = encoder.to(device).eval()

with torch.no_grad():
    tokens = encoder(images_vjepa)

print("V-JEPA tokens:", tokens.shape)

# Depth Anything Depth Estimation

In [ ]:
sys.path.append("../external/Depth-Anything-V2")
from depth_anything_v2.dpt import DepthAnythingV2

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# see https://github.com/DepthAnything/Depth-Anything-V2#pre-trained-models
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = DEPTH_ANYTHING_MODEL

da_model = DepthAnythingV2(**model_configs[encoder])
da_model.load_state_dict(torch.load(f'../checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu'))
da_model = da_model.to(device).eval()

## Show some Predictions of Train Data

In [ ]:
batch = next(iter(train_loader))
images = batch['image'].to(device)
print('input images:', images.shape)

images_da = images / 255.0

with torch.no_grad():
    pred_depth = da_model(images_da)

print('Depth Anything output:', pred_depth.shape)

images = batch["image"]
gt_depths = batch["depth"]
pred_depths = pred_depth

batch_size = BATCH_SIZE

fig, axes = plt.subplots(3, batch_size, figsize=(4 * batch_size, 12))

for i in range(batch_size):
    rgb = images[i].detach().cpu().permute(1, 2, 0).numpy() / 255.0
    gt_depth = gt_depths[i, 0].detach().cpu().numpy()
    da_depth = pred_depths[i].detach().cpu().numpy()
    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"RGB {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(gt_depth, cmap='viridis')
    axes[1, i].set_title(f"GT depth {i}")
    axes[1, i].axis("off")
    axes[2, i].imshow(da_depth, cmap="viridis")
    axes[2, i].set_title(f"DA pred {i}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
rows = []
da_model.eval()

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        image_ids = batch["id"]

        images_da = images / 255.0
        pred_depths = da_model(images_da)

        if pred_depths.ndim == 4:
            pred_depths = pred_depths.squeeze(1)

        pred_depths = pred_depths.detach().cpu().numpy()

        for depth, image_id in zip(pred_depths, image_ids):
            rows.append({"id": f"{image_id}_depth","Depths": encode_depth(depth)})

df = save_submission(rows, "../submission.csv")